# Smoke test — inventory-streaming

Weryfikacja infrastruktury: Kafka, PostgreSQL, Redis.

Uruchom po `docker compose up -d --build` w JupyterLab (`http://localhost:8999`).

In [ ]:
import json
import os
from datetime import datetime

import psycopg2
import redis
from kafka import KafkaConsumer, KafkaProducer

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_INTERNAL", "broker:9092")
POSTGRES = {
    "host": os.environ.get("POSTGRES_HOST", "postgres"),
    "port": os.environ.get("POSTGRES_PORT", "5432"),
    "dbname": os.environ.get("POSTGRES_DB", "inventory"),
    "user": os.environ.get("POSTGRES_USER", "inventory"),
    "password": os.environ.get("POSTGRES_PASSWORD", "inventory"),
}
REDIS_HOST = os.environ.get("REDIS_HOST", "redis")
REDIS_PORT = int(os.environ.get("REDIS_PORT", "6379"))

TEST_EVENT = {
    "event_id": "SALE-SMOKE-001",
    "product_id": "P001",
    "quantity": 1,
    "unit_price": 49.99,
    "store_id": "WAW-01",
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}

print(f"Kafka: {KAFKA_BOOTSTRAP}")
print(f"PostgreSQL: {POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['dbname']}")
print(f"Redis: {REDIS_HOST}:{REDIS_PORT}")

In [ ]:
# 1. Wyślij wiadomość testową na sales.events
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
future = producer.send("sales.events", key=b"P001", value=TEST_EVENT)
future.get(timeout=10)
producer.flush()
print(f"Wysłano: {TEST_EVENT}")

In [ ]:
# 2. Odczytaj wiadomość konsumentem testowym
consumer = KafkaConsumer(
    "sales.events",
    bootstrap_servers=KAFKA_BOOTSTRAP,
    group_id="infra-smoke-test",
    auto_offset_reset="earliest",
    consumer_timeout_ms=15000,
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
)

received = None
for message in consumer:
    if message.value.get("event_id") == TEST_EVENT["event_id"]:
        received = message.value
        break

consumer.close()
assert received is not None, "Nie odebrano wiadomości testowej z sales.events"
print(f"Odebrano: {received}")

In [ ]:
# 3. Sprawdź połączenie z PostgreSQL
with psycopg2.connect(**POSTGRES) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1")
        assert cur.fetchone()[0] == 1
        cur.execute("SELECT COUNT(*) FROM products")
        product_count = cur.fetchone()[0]

print(f"PostgreSQL OK — produktów w seed: {product_count}")
assert product_count >= 1

In [ ]:
# 4. Zapisz i odczytaj klucz testowy w Redis
r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, decode_responses=True)
assert r.ping()

test_key = "infra:smoke_test"
r.set(test_key, "ok", ex=60)
assert r.get(test_key) == "ok"
r.delete(test_key)

print("Redis OK")
print("\n=== Smoke test PASSED ===")